# Data Preprocessing

This notebook prepares the raw transaction data for machine learning.
The preprocessing steps are performed only when required by the dataset.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/financial_fraud_dataset.csv")

print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (10000, 10)


In [2]:
#Before applying any preprocessing, we check for missing values and duplicate records.

print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

Missing values:
transaction_id           0
timestamp                0
amount                   0
merchant_category        0
customer_id              0
customer_age             0
customer_location        0
device_type              0
previous_transactions    0
is_fraud                 0
dtype: int64

Duplicate rows: 0


In [3]:
df.dtypes

transaction_id            object
timestamp                 object
amount                   float64
merchant_category         object
customer_id               object
customer_age               int64
customer_location         object
device_type               object
previous_transactions      int64
is_fraud                   int64
dtype: object

In [ ]:
#The transaction timestamp is converted into datetime format so that time-based information can be used during later analysis.

df["timestamp"] = pd.to_datetime(df["timestamp"])

print(df["timestamp"].dtype)

datetime64[ns]


In [5]:
# Identify numerical and categorical columns

numerical_columns = df.select_dtypes(include=np.number).columns.tolist()
categorical_columns = df.select_dtypes(include="object").columns.tolist()

print("Numerical columns:")
print(numerical_columns)

print("\nCategorical columns:")
print(categorical_columns)

Numerical columns:
['amount', 'customer_age', 'previous_transactions', 'is_fraud']

Categorical columns:
['transaction_id', 'merchant_category', 'customer_id', 'customer_location', 'device_type']


In [6]:
#The target variable and identifier columns are separated from the predictive features before applying further preprocessing.

target_column = "is_fraud"

identifier_columns = ["transaction_id", "customer_id"]

feature_columns = [
    col for col in df.columns
    if col not in [target_column] + identifier_columns
]

print("Feature columns:")
print(feature_columns)

print("\nTarget column:")
print(target_column)

print("\nIdentifier columns:")
print(identifier_columns)

Feature columns:
['timestamp', 'amount', 'merchant_category', 'customer_age', 'customer_location', 'device_type', 'previous_transactions']

Target column:
is_fraud

Identifier columns:
['transaction_id', 'customer_id']


In [7]:
#Categorical variables are converted into numerical form using one-hot encoding. This allows machine learning models to process categorical transaction attributes.


categorical_features = [
    "merchant_category",
    "customer_location",
    "device_type"
]

df_encoded = pd.get_dummies(
    df,
    columns=categorical_features,
    drop_first=True
)

print("Shape after encoding:", df_encoded.shape)
df_encoded.head()

Shape after encoding: (10000, 17)


,transaction_id,timestamp,amount,customer_id,customer_age,previous_transactions,is_fraud,merchant_category_entertainment,merchant_category_fashion,merchant_category_fuel,merchant_category_grocery,customer_location_FL,customer_location_IL,customer_location_NY,customer_location_TX,device_type_mobile,device_type_tablet
0,916306a3-f7e2-4371-992a-38ab8030b680,2022-01-02 06:53:04,46.93,bef45806-a255-4cdb-a45e-012d533eee34,48,2,0,False,False,True,False,False,False,True,False,False,True
1,2d922a41-edd9-4649-81f0-c599db4b923d,2024-09-05 14:18:29,301.01,f600bad0-d244-42c4-9db3-1115214dd1c4,58,4,0,False,False,False,False,False,True,False,False,True,False
2,9d2ec6cd-f8ed-4c68-9ec7-1450c30b7fe0,2022-12-07 06:24:09,131.67,050c0996-fd20-4ce0-94ec-b1bbc1f3c229,23,2,0,True,False,False,False,False,False,False,True,False,True
3,f65f1e63-1b91-4a79-95e4-4dd825992461,2022-08-19 03:58:47,91.29,71693a5a-7103-4876-9e03-0fcb64ba3e28,69,6,0,False,True,False,False,True,False,False,False,False,False
4,e6a8161d-4fdc-4483-9b61-2c8c86567e78,2023-04-19 12:55:52,16.96,18ed5ca8-d0eb-4ccf-b4d3-fa37f15b89c7,48,0,0,False,True,False,False,False,False,False,True,False,True


In [9]:
# The input features are separated from the target variable. The target variable `is_fraud` is kept separately for model training and evaluation.

X = df_encoded.drop(columns=["is_fraud"])
y = df_encoded["is_fraud"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (10000, 16)
Target shape: (10000,)


Final model preprocessing is performed after temporal splitting to avoid data leakage.